In [0]:
%sql CREATE SCHEMA IF NOT EXISTS dbr_dev_ua5816bd.natalkamartinuk55;


In [0]:

df_raw = (
    spark.read
    .format("csv")
    .option("header", "true")        
    .option("inferSchema", "true")   
    .option("multiLine", "true")     
    .option("escape", "\"")          
    .load("/Volumes/dbr_dev_ua5816bd/natalkamartinuk55/raw_data/USvideos.csv")
)

(
    df_raw.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("dbr_dev_ua5816bd.natalkamartinuk55.youtube_videos")
)

display(spark.table("dbr_dev_ua5816bd.natalkamartinuk55.youtube_videos").limit(5))


In [0]:
from pyspark.sql.functions import col, desc, sum, count, avg

df_videos = spark.table("dbr_dev_ua5816bd.natalkamartinuk55.youtube_videos")

df_top_channels =(
    df_videos
    .select("channel_title", "views", "likes", "comment_count")
    .filter(col("views") > 500000)
    .groupBy("channel_title")
    .agg(
        count("*").alias("total_videos"),
        sum("views").alias("total_views"),
        avg("likes").alias("avg_likes")
    )
    .orderBy(desc("total_views"))
)

display(df_top_channels.limit(10))

In [0]:
import requests
from pyspark.sql.functions import col, round

response = requests.get("https://bank.gov.ua/NBUStatService/v1/statdirectory/exchange?json")
data_json = response.json()

df_rates = spark.createDataFrame(data_json)
(
    df_rates.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("dbr_dev_ua5816bd.natalkamartinuk55.currency_rates")
)

df_usd = df_rates.filter(col("cc") == "USD").select(col("rate").alias("usd_to_uah_rate"))

df_final = (
    df_top_channels
    .crossJoin(df_usd)
    .withColumn("est_revenue_usd", round((col("total_views") / 1000) * 1.5, 2))
    .withColumn("est_revenue_uah", round(col("est_revenue_usd") * col("usd_to_uah_rate"), 2))
)

display(df_final.limit(10))

Databricks visualization. Run in Databricks to view.

In [0]:
(
    df_final.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("dbr_dev_ua5816bd.natalkamartinuk55.final_channel_analytics")
)

%md
## Delta Lake Architecture & Key Benefits

Delta Lake is an open-source storage framework that brings reliability, performance, and ACID transactions to data lakes. While raw data formats like CSV or standard Parquet can be prone to data corruption and inconsistency, Delta Lake provides enterprise-grade data management features.

Here are the primary advantages of utilizing Delta Lake in modern data pipelines:

---

### 1.ACID Transactions
* **Atomic & Isolated Operations:** Delta Lake relies on a write-ahead transaction log (`_delta_log`), ensuring that every write, update, or append operation is all-or-nothing.
* **Corrupted Data Prevention:** If a cluster crashes midway during an ingestion job, partial writes are never exposed to consumers, guaranteeing consistent and reliable reads.

### 2.Time Travel & Version History
* **Automatic Versioning:** Every transaction creates an immutable new snapshot version of the table (e.g., `v0`, `v1`, `v2`).
* **Effortless Rollbacks & Auditing:** Developers can query historical states or restore tables instantly using the `VERSION AS OF` or `TIMESTAMP AS OF` syntax without maintaining separate manual backups.

### 3.Schema Enforcement & Evolution
* **Schema Enforcement (Guardrails):** Prevents bad or malformed data from corrupting existing tables by rejecting incoming DataFrames that mismatch predefined data types or column layouts.
* **Schema Evolution:** Allows intentional schema updates (adding new columns dynamically) safely via the `.option("mergeSchema", "true")` configuration.